# Пара 4+. Граф совстречаемости персонажей (NetworkX в Colab)

Берём «Чистый понедельник», ищем имена из учебного списка, строим граф: ребро = совместное появление в окне N предложений.

Colab спокойно рисует NetworkX + matplotlib.


In [ ]:
!pip -q install networkx


In [ ]:
import re, requests, math
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations

url = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/bunin-temnye-allei/txt/chistyj-ponedelnik.txt"
text = requests.get(url, timeout=60).text

# учебный список (не NER): нормализованные узлы и варианты
NODES = {
    "он": ["он ", "его ", "ему "],  # осторожно: шумно; лучше конкретные имена
    "она": ["она ", "её ", "ей "],
}
# Более строгий список под рассказ:
NODES = {
    "Рассказчик": ["я ", "мне ", "меня ", "мной"],
    "Она": ["она ", "её ", "ей ", "нею"],
    "Егор": ["егорушка", "егор"],
    "Москва": ["москв"],
    "Ордынка": ["ордын"],
    "Марфо-Мариинская": ["марфо", "мариин"],
}

sents = re.split(r"(?<=[.!?…])\s+", text)
sents = [s for s in sents if s.strip()]
print("предложений:", len(sents))

def present(sent, variants):
    low = " " + sent.lower() + " "
    return any(v in low or sent.lower().startswith(v.strip()) or v.strip() in sent.lower() for v in variants)

window = 2  # соседние предложения
edges = {}
for i in range(len(sents)):
    chunk = " ".join(sents[i:i+window])
    here = [n for n, vars_ in NODES.items() if present(chunk, vars_)]
    for a, b in combinations(sorted(set(here)), 2):
        edges[(a,b)] = edges.get((a,b), 0) + 1

G = nx.Graph()
for (a,b), w in edges.items():
    if w >= 2:  # порог против шума
        G.add_edge(a, b, weight=w)

print("узлов", G.number_of_nodes(), "рёбер", G.number_of_edges())
print("степени:", dict(G.degree()))

plt.figure(figsize=(8,6))
pos = nx.spring_layout(G, seed=7)
weights = [G[u][v]["weight"] for u,v in G.edges()]
nx.draw_networkx_nodes(G, pos, node_size=1200, node_color="#99f6e4")
nx.draw_networkx_labels(G, pos, font_size=10)
nx.draw_networkx_edges(G, pos, width=[0.5+w*0.4 for w in weights])
nx.draw_networkx_edge_labels(G, pos, edge_labels={(u,v):d["weight"] for u,v,d in G.edges(data=True)}, font_size=8)
plt.title("Совстречаемость (окно=2 предл., вес≥2) · Чистый понедельник")
plt.axis("off")
plt.tight_layout()
plt.show()
plt.savefig("bunin_cooccur_graph.png", dpi=150, bbox_inches="tight")
print("saved bunin_cooccur_graph.png")


### Разбор
1. Какое ребро спорное? Почему окно/порог его создали?
2. Измените `window` или порог `w >= 2` и сравните граф.
3. Правило ребра — часть метода: запишите его одной фразой.
